<a href="https://colab.research.google.com/github/Chaospossum/cv-robustness-thesis/blob/main/run_experiments1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training for Robustness  Full Experiment Runner

**Capstone:** Comparing Degradation Augmentation Strategies in CNN Image Classification

This notebook runs the full experimental matrix (50 runs) on Google Colab GPU,
saving results to Google Drive after each completed run.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Create project directory on Drive for persistent results
import os
DRIVE_DIR = '/content/drive/MyDrive/capstone_robustness'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'baseline'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'augmented'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'baseline', 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'augmented', 'checkpoints'), exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')

Drive dir: /content/drive/MyDrive/capstone_robustness


In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


## Core Code
All code is self-contained below — no external imports needed.

In [ ]:
import io
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader
from PIL import Image, ImageFilter, ImageEnhance
from scipy.ndimage import convolve1d


# ═══════════════════════════════════════════════════════════════════════════
# DEGRADATION GRID
# ═══════════════════════════════════════════════════════════════════════════
DEG_GRID = {
    'gaussian_noise': [0.02, 0.05, 0.10],
    'gaussian_blur':  [3, 5, 7],
    'motion_blur':    [3, 5, 7],
    'jpeg':           [50, 30, 10],
    'contrast':       [0.8, 0.6, 0.4],
    'darkening':      [0.8, 0.6, 0.4],
}


# ═══════════════════════════════════════════════════════════════════════════
# DEGRADATION FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════
def apply_gaussian_noise(img_tensor, sigma):
    noise = torch.randn_like(img_tensor) * sigma
    return torch.clamp(img_tensor + noise, 0.0, 1.0)

def apply_gaussian_blur(img, kernel_size):
    return img.filter(ImageFilter.GaussianBlur(radius=kernel_size // 2))

def apply_motion_blur(img, kernel_size):
    img_np = np.array(img).astype(np.float64)
    kernel = np.ones(kernel_size) / kernel_size
    blurred = convolve1d(img_np, kernel, axis=1, mode='nearest')
    return Image.fromarray(np.clip(blurred, 0, 255).astype(np.uint8))

def apply_jpeg_compression(img, quality):
    buffer = io.BytesIO()
    img.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    return Image.open(buffer).convert('RGB')

def apply_contrast(img, factor):
    return ImageEnhance.Contrast(img).enhance(factor)

def apply_darkening(img, factor):
    return ImageEnhance.Brightness(img).enhance(factor)

_DEG_FNS = {
    'gaussian_noise': (apply_gaussian_noise, 'tensor'),
    'gaussian_blur':  (apply_gaussian_blur,  'pil'),
    'motion_blur':    (apply_motion_blur,    'pil'),
    'jpeg':           (apply_jpeg_compression, 'pil'),
    'contrast':       (apply_contrast,       'pil'),
    'darkening':      (apply_darkening,      'pil'),
}

def apply_degradation(img_tensor, deg_type, severity_value):
    fn, input_type = _DEG_FNS[deg_type]
    if input_type == 'tensor':
        return fn(img_tensor, severity_value)
    img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    pil_img = Image.fromarray(img_np)
    pil_img = fn(pil_img, severity_value)
    img_np = np.array(pil_img).astype(np.float32) / 255.0
    return torch.from_numpy(img_np).permute(2, 0, 1)


# ═══════════════════════════════════════════════════════════════════════════
# CURRICULUM SCHEDULER
# ═══════════════════════════════════════════════════════════════════════════
def get_curriculum_severity(deg_type, epoch, total_epochs=200):
    levels = DEG_GRID[deg_type]
    ramp_end = total_epochs // 2
    if epoch <= 1:
        progress = 0.0
    elif epoch >= ramp_end:
        progress = 1.0
    else:
        progress = (epoch - 1) / (ramp_end - 1)
    mild, hard = levels[0], levels[-1]
    value = mild + progress * (hard - mild)
    if deg_type in ('gaussian_blur', 'motion_blur'):
        value = int(round(value))
        if value % 2 == 0:
            value += 1
    return value


# ═══════════════════════════════════════════════════════════════════════════
# MODELS
# ═══════════════════════════════════════════════════════════════════════════
def resnet18_cifar10():
    model = models.resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

def mobilenetv2_cifar10():
    model = models.mobilenet_v2(weights=None, num_classes=10)
    first_conv = model.features[0][0]
    model.features[0][0] = nn.Conv2d(
        first_conv.in_channels, first_conv.out_channels,
        kernel_size=first_conv.kernel_size, stride=1,
        padding=first_conv.padding, bias=False,
    )
    return model

def get_model(name):
    if name == 'resnet18':
        return resnet18_cifar10()
    elif name == 'mobilenetv2':
        return mobilenetv2_cifar10()
    raise ValueError(f'Unknown model: {name}')


# ═══════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════
CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
CIFAR10_STD  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
SEEDS = [0, 42, 123, 456, 789]
EPOCHS = 200
BATCH_SIZE = 128
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
MILESTONES = [100, 150]
GAMMA = 0.1

print('All code loaded.')

All code loaded.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_train_loader():
    transform = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
    ])
    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )
    return DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)

def get_test_loader():
    transform = T.ToTensor()
    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )
    return DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

def normalize_batch(images, device):
    mean = CIFAR10_MEAN.to(device)
    std = CIFAR10_STD.to(device)
    return (images - mean) / std

# Augmentation functions
def augment_single(images, deg_type):
    levels = DEG_GRID[deg_type]
    return torch.stack([apply_degradation(img, deg_type, random.choice(levels)) for img in images])

def augment_mixed(images):
    deg_types = list(DEG_GRID.keys())
    out = []
    for img in images:
        dt = random.choice(deg_types)
        out.append(apply_degradation(img, dt, random.choice(DEG_GRID[dt])))
    return torch.stack(out)

def augment_curriculum(images, deg_type, epoch):
    sev = get_curriculum_severity(deg_type, epoch, EPOCHS)
    return torch.stack([apply_degradation(img, deg_type, sev) for img in images])

print('Helpers loaded.')

Helpers loaded.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EVALUATION
# ═══════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate_accuracy(model, device, deg_type=None, severity_value=None):
    """Evaluate on clean or degraded test set. Returns accuracy %."""
    model.eval()
    loader = get_test_loader()
    correct = total = 0
    for images, labels in loader:
        if deg_type is not None:
            images = torch.stack([apply_degradation(img, deg_type, severity_value) for img in images])
        images = normalize_batch(images.to(device), device)
        labels = labels.to(device)
        _, predicted = model(images).max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / total

def full_eval(model, device, model_name, strategy, seed, deg_trained='none'):
    """Full robustness evaluation: clean + 18 conditions."""
    results = []
    acc = evaluate_accuracy(model, device)
    results.append({'model': model_name, 'strategy': strategy, 'deg_type_trained': deg_trained,
                    'seed': seed, 'eval_degradation': 'clean', 'eval_severity': 0, 'accuracy': acc})
    print(f'  Clean: {acc:.2f}%')

    for deg_name, levels in DEG_GRID.items():
        for sev_idx, sev_val in enumerate(levels, 1):
            acc = evaluate_accuracy(model, device, deg_name, sev_val)
            results.append({'model': model_name, 'strategy': strategy, 'deg_type_trained': deg_trained,
                            'seed': seed, 'eval_degradation': deg_name, 'eval_severity': sev_idx, 'accuracy': acc})
            print(f'  {deg_name} sev{sev_idx}: {acc:.2f}%')
    return results

print('Evaluation loaded.')

Evaluation loaded.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# TRAINING + SAVE TO DRIVE
# ═══════════════════════════════════════════════════════════════════════════

def run_experiment(strategy, model_name, seed, deg_type='none', drive_dir=DRIVE_DIR):
    """Train one model, evaluate, save results + checkpoint to Drive."""

    # Build file paths
    subdir = 'baseline' if strategy == 'baseline' else 'augmented'
    if strategy in ('single', 'curriculum'):
        csv_name = f'{model_name}_{strategy}_{deg_type}_robustness.csv'
        ckpt_name = f'{model_name}_{strategy}_{deg_type}_seed{seed}.pt'
    else:
        csv_name = f'{model_name}_{strategy}_robustness.csv'
        ckpt_name = f'{model_name}_{strategy}_seed{seed}.pt'

    csv_path = os.path.join(drive_dir, subdir, csv_name)
    ckpt_path = os.path.join(drive_dir, subdir, 'checkpoints', ckpt_name)

    # Skip if this seed already exists in CSV (fully completed)
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        if seed in existing['seed'].values:
            print(f'SKIP: {strategy}/{model_name}/seed{seed}/{deg_type} already done')
            return None

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'\n{"="*60}')
    print(f'Strategy: {strategy} | Model: {model_name} | Seed: {seed} | Deg: {deg_type}')
    print(f'Device: {device}')
    print(f'{"="*60}')

    model = get_model(model_name).to(device)

    # If checkpoint exists but CSV doesn't, skip training and just re-evaluate
    if os.path.exists(ckpt_path):
        print(f'RECOVER: Loading checkpoint (training was done, eval crashed)')
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
    else:
        # Full training
        set_seed(seed)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=MILESTONES, gamma=GAMMA)
        loader = get_train_loader()

        start = time.time()
        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0
            total_samples = 0

            for images, labels in loader:
                if strategy == 'single':
                    images = augment_single(images, deg_type)
                elif strategy == 'mixed':
                    images = augment_mixed(images)
                elif strategy == 'curriculum':
                    images = augment_curriculum(images, deg_type, epoch)

                images = normalize_batch(images.to(device), device)
                labels = labels.to(device)

                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * labels.size(0)
                total_samples += labels.size(0)

            scheduler.step()
            if epoch % 20 == 0 or epoch == 1:
                elapsed = time.time() - start
                avg_loss = total_loss / total_samples
                print(f'  Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | Time: {elapsed:.0f}s')

        train_time = time.time() - start
        print(f'Training complete in {train_time:.0f}s ({train_time/60:.1f}min)')

        # Save checkpoint to Drive
        os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
        torch.save(model.state_dict(), ckpt_path)
        print(f'Checkpoint: {ckpt_path}')

    # Evaluate
    print('Evaluating robustness...')
    results = full_eval(model, device, model_name, strategy, seed, deg_type)

    # Save/append CSV to Drive
    df = pd.DataFrame(results)
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        df = pd.concat([existing, df], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print(f'Results saved: {csv_path}')

    return df

## Configuration

Edit the cell below to control which experiments to run.  
Set `RUN_ALL = True` to run the full matrix, or customize the lists.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit this cell
# ═══════════════════════════════════════════════════════════════════════════
RUN_ALL = True  # Set True to run full 50-experiment matrix

# Or customize:
# STRATEGIES = ['baseline', 'mixed', 'single', 'curriculum']
# MODEL_NAMES = ['resnet18', 'mobilenetv2']
# SINGLE_DEGS = ['gaussian_blur', 'gaussian_noise']  # for single + curriculum

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BUILD EXPERIMENT LIST
# ═══════════════════════════════════════════════════════════════════════════
experiments = []

MODELS_LIST = ['resnet18', 'mobilenetv2']
SINGLE_DEGS = ['gaussian_blur', 'gaussian_noise']

if RUN_ALL:
    # Baseline: 2 × 5 = 10
    for m in MODELS_LIST:
        for s in SEEDS:
            experiments.append(('baseline', m, s, 'none'))
    # Mixed: 2 × 5 = 10
    for m in MODELS_LIST:
        for s in SEEDS:
            experiments.append(('mixed', m, s, 'none'))
    # Single: 2 × 2 × 5 = 20
    for d in SINGLE_DEGS:
        for m in MODELS_LIST:
            for s in SEEDS:
                experiments.append(('single', m, s, d))
    # Curriculum: 2 × 2 × 5 = 20
    for d in SINGLE_DEGS:
        for m in MODELS_LIST:
            for s in SEEDS:
                experiments.append(('curriculum', m, s, d))

print(f'Total experiments: {len(experiments)}')

Total experiments: 60


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN ALL EXPERIMENTS
# ═══════════════════════════════════════════════════════════════════════════
# Already-completed runs are automatically skipped.
# If Colab disconnects, just re-run this cell to resume.

for i, (strategy, model_name, seed, deg_type) in enumerate(experiments, 1):
    print(f'\n[{i}/{len(experiments)}]')
    run_experiment(strategy, model_name, seed, deg_type)

print('\n\nALL EXPERIMENTS COMPLETE!')


[1/60]
SKIP: baseline/resnet18/seed0/none already done

[2/60]
SKIP: baseline/resnet18/seed42/none already done

[3/60]
SKIP: baseline/resnet18/seed123/none already done

[4/60]
SKIP: baseline/resnet18/seed456/none already done

[5/60]
SKIP: baseline/resnet18/seed789/none already done

[6/60]
SKIP: baseline/mobilenetv2/seed0/none already done

[7/60]
SKIP: baseline/mobilenetv2/seed42/none already done

[8/60]
SKIP: baseline/mobilenetv2/seed123/none already done

[9/60]
SKIP: baseline/mobilenetv2/seed456/none already done

[10/60]
SKIP: baseline/mobilenetv2/seed789/none already done

[11/60]
SKIP: mixed/resnet18/seed0/none already done

[12/60]
SKIP: mixed/resnet18/seed42/none already done

[13/60]
SKIP: mixed/resnet18/seed123/none already done

[14/60]
SKIP: mixed/resnet18/seed456/none already done

[15/60]
SKIP: mixed/resnet18/seed789/none already done

[16/60]
SKIP: mixed/mobilenetv2/seed0/none already done

[17/60]
SKIP: mixed/mobilenetv2/seed42/none already done

[18/60]
SKIP: mix

100%|██████████| 170M/170M [00:12<00:00, 13.1MB/s]


  Epoch   1/200 | Loss: 1.9182 | Time: 10s
  Epoch  20/200 | Loss: 0.5263 | Time: 172s
  Epoch  40/200 | Loss: 0.4738 | Time: 342s
  Epoch  60/200 | Loss: 0.4577 | Time: 512s
  Epoch  80/200 | Loss: 0.4478 | Time: 683s
  Epoch 100/200 | Loss: 0.4412 | Time: 852s
  Epoch 120/200 | Loss: 0.1078 | Time: 1022s
  Epoch 140/200 | Loss: 0.1036 | Time: 1191s
  Epoch 160/200 | Loss: 0.0263 | Time: 1362s
  Epoch 180/200 | Loss: 0.0165 | Time: 1532s
  Epoch 200/200 | Loss: 0.0147 | Time: 1702s
Training complete in 1702s (28.4min)
Checkpoint: /content/drive/MyDrive/capstone_robustness/augmented/checkpoints/resnet18_single_gaussian_noise_seed456.pt
Evaluating robustness...
  Clean: 93.15%
  gaussian_noise sev1: 93.02%
  gaussian_noise sev2: 92.37%
  gaussian_noise sev3: 90.48%
  gaussian_blur sev1: 66.69%
  gaussian_blur sev2: 27.90%
  gaussian_blur sev3: 21.49%
  motion_blur sev1: 85.54%
  motion_blur sev2: 67.32%
  motion_blur sev3: 52.16%
  jpeg sev1: 89.36%
  jpeg sev2: 87.06%
  jpeg sev3: 70.9

## Verify Results
Check what's been completed so far.

In [ ]:
import glob

csvs = glob.glob(os.path.join(DRIVE_DIR, '**', '*_robustness.csv'), recursive=True)
print(f'Found {len(csvs)} result files:\n')
total_seeds = 0
for f in sorted(csvs):
    df = pd.read_csv(f)
    n_seeds = df['seed'].nunique()
    total_seeds += n_seeds
    print(f'  {os.path.basename(f)}: {n_seeds} seeds')
print(f'\nTotal completed runs: {total_seeds}/50')

Found 12 result files:

  mobilenetv2_curriculum_gaussian_blur_robustness.csv: 5 seeds
  mobilenetv2_curriculum_gaussian_noise_robustness.csv: 5 seeds
  mobilenetv2_mixed_robustness.csv: 5 seeds
  mobilenetv2_single_gaussian_blur_robustness.csv: 5 seeds
  mobilenetv2_single_gaussian_noise_robustness.csv: 5 seeds
  resnet18_curriculum_gaussian_blur_robustness.csv: 5 seeds
  resnet18_curriculum_gaussian_noise_robustness.csv: 5 seeds
  resnet18_mixed_robustness.csv: 5 seeds
  resnet18_single_gaussian_blur_robustness.csv: 5 seeds
  resnet18_single_gaussian_noise_robustness.csv: 5 seeds
  mobilenetv2_baseline_robustness.csv: 5 seeds
  resnet18_baseline_robustness.csv: 5 seeds

Total completed runs: 60/50
